# ML-09 — Validation Audit and Slice Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sohila-Khaled-Abbas/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

Auditing model performance across client domain slices and evaluating Precision@K.

## 1. Grouped Client Holdout Validation

*Evaluating Precision@50 and Precision@100 on unseen client domains.*

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import RandomForestClassifier

csv_path = "../../data/raw/content_refresh_anonymized.csv"
if not os.path.exists(csv_path):
    csv_path = "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(csv_path)
df["pos_clean"] = np.where(df["avg_position"] == 0, 50.0, df["avg_position"])
df["click_drop"] = np.maximum(0, df["clicks_prev_30d"] - df["clicks_last_30d"])
df["target_binary"] = (df["click_drop"] > df["click_drop"].median()).astype(int)

clients = df["client_id"].unique()
np.random.seed(42)
train_clients = np.random.choice(clients, size=int(len(clients)*0.8), replace=False)

train_df = df[df["client_id"].isin(train_clients)].copy()
test_df = df[~df["client_id"].isin(train_clients)].copy()

features = ["impressions_90d", "clicks_prev_30d", "pos_clean", "content_age_days", "ctr"]
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(train_df[features], train_df["target_binary"])

test_df["score"] = rf.predict_proba(test_df[features])[:, 1]
ranked = test_df.sort_values(by="score", ascending=False)

p50 = test_df["target_binary"].loc[ranked.index[:50]].mean()
p100 = test_df["target_binary"].loc[ranked.index[:100]].mean()
print(f"Holdout Precision@50: {p50:.1%}")
print(f"Holdout Precision@100: {p100:.1%}")


Holdout Precision@50: 100.0%
Holdout Precision@100: 100.0%


## 2. Slice Analysis Across Client Domains

*Performance breakdown across individual client holdout domains.*

In [2]:
slice_perf = test_df.groupby("client_id").agg(
    total_pages=("content_id", "count"),
    mean_score=("score", "mean"),
    target_rate=("target_binary", "mean")
).reset_index()
print("=== CLIENT HOLDOUT SLICE PERFORMANCE ===")
print(slice_perf.head(7).to_string(index=False))


=== CLIENT HOLDOUT SLICE PERFORMANCE ===
        client_id  total_pages  mean_score  target_rate
client_2c624232cd          649    0.088511     0.104777
client_349c41201b          763    0.429155     0.382700
client_6208ef0f77         3681    0.365390     0.282804
client_624b60c58c          337    0.096391     0.089021
client_bdd2d3af3a           44    0.126125     0.068182
client_d4735e3a26         1106    0.107690     0.108499
client_f74efabef1         1031    0.218265     0.203686


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`